# 00 — Colab setup: dataset, cleaning, split and EDA

Run this notebook **once per Colab session** before any training notebook.

**How to use:** `Runtime > Run all`. No configuration is needed.
- The dataset is downloaded with `kagglehub`. If Kaggle asks for authentication, a login
  box appears in the output: enter your Kaggle credentials there (they are not saved in the notebook).
- Alternative without Kaggle: put the dataset `.zip` (downloaded from the Kaggle page) in
  `MyDrive/road-damage-tl/datasets/` — it will be used instead of downloading.
- A GPU is not needed for this notebook (it is for training notebooks).

**Storage strategy**
- The Colab disk (`/content`) is erased when the session ends.
- The dataset archive and everything that must survive (checkpoints, MLflow runs, results)
  is kept on **Google Drive** in `MyDrive/road-damage-tl/`.
- Images are unzipped to the local disk `/content/data/raw` for every session: reading thousands
  of small files directly from Drive is very slow.

In [ ]:
REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
KAGGLE_DATASET = "prudhvignv/road-damage-classification-and-assessment"

DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"   # persistent storage
REPO_DIR = "/content/road-damage-tl"                   # code (cloned from GitHub)
RAW_DIR = "/content/data/raw"                          # images (fast local disk)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ("datasets", "checkpoints", "mlruns", "results"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

## 2. Clone (or update) the repository

If the repository is **private**, create a GitHub *fine-grained personal access token*
(read-only on this repo), store it as the Colab Secret `GITHUB_TOKEN`, and it will be used automatically.

In [ ]:
import os, subprocess

url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # public repository: no token needed

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

%cd {REPO_DIR}
!git log --oneline -3

## 3. Install dependencies

Torch, torchvision, pandas, scikit-learn and matplotlib are pre-installed on Colab.

In [ ]:
!pip install -q timm albumentations grad-cam mlflow kagglehub

## 4. Get the dataset

1. If a `.zip` is already in `MyDrive/road-damage-tl/datasets/` (cached from a previous session,
   or uploaded manually), it is simply unzipped.
2. Otherwise it is downloaded from Kaggle with `kagglehub`, then cached on Drive as a zip,
   so the download happens only once.

In [ ]:
import glob, os, shutil, subprocess

os.makedirs(RAW_DIR, exist_ok=True)
zips = sorted(glob.glob(f"{DRIVE_ROOT}/datasets/*.zip"))

if zips:
    print("Using archive from Drive:", zips[0])
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", RAW_DIR], check=True)
else:
    import kagglehub
    try:
        path = kagglehub.dataset_download(KAGGLE_DATASET)
    except Exception as e:
        print("Anonymous download failed:", e)
        print("-> Log in with your Kaggle account below.")
        kagglehub.login()
        path = kagglehub.dataset_download(KAGGLE_DATASET)
    shutil.copytree(path, RAW_DIR, dirs_exist_ok=True)
    cache = f"{DRIVE_ROOT}/datasets/road-damage-dataset"
    shutil.make_archive(cache, "zip", RAW_DIR)
    print("Downloaded and cached on Drive:", cache + ".zip")

n_files = sum(len(files) for _, _, files in os.walk(RAW_DIR))
print(f"{n_files} files in {RAW_DIR}")

## 5. Inspect the real structure of the dataset

`make_splits.py` assumes **one folder per class** (the class is the image's parent folder).
Check it here before going further.

In [ ]:
!find {RAW_DIR} -maxdepth 3 -type d | head -30
print()
!find {RAW_DIR} -type f | sed 's/.*\.//' | sort | uniq -c

## 6. Clean the data and create the reproducible split (70/15/15, seed 42)

In [ ]:
!python -m src.data.make_splits --raw {RAW_DIR}

## 7. Exploratory data analysis

In [ ]:
!python -m src.data.eda --raw {RAW_DIR}

from IPython.display import Image, Markdown, display
display(Markdown(open("results/eda/dataset_table.md").read()))
for f in ("class_distribution", "split_distribution", "image_sizes", "samples"):
    display(Image(f"results/eda/{f}.png", width=700))

## 8. Save the outputs

- Copy to Drive (backup).
- Download a zip, then **commit `data/splits/` and `results/eda/` from your PC**
  (the split CSVs must be versioned so that the whole team uses the same images).

In [ ]:
!cp -r data/splits results/eda "{DRIVE_ROOT}/results/"
!zip -q -r /content/splits_and_eda.zip data/splits results/eda

from google.colab import files
files.download("/content/splits_and_eda.zip")